# Identificacion de sentimientos a partir de videos.
A partir de un listado de videos guardados en google Drive.




### 1. Pasar de video a Fotos.

### Instalacion de librerias

In [1]:
pip install deepface

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 18.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.0 MB/s eta 0:00:00
  Created wheel for fire: filename=fire-0.5.0-py2.py3-none-any.whl size=116932 sha256=23a8b695699c60c5b465878e526e98a972ba4218fc44b8f8a2e074b5611d979c
  Stored in directory: /root/.cache/pip/wheels/90/d4/f7/9404e5db0116bd4d43e5666eaa3e70ab53723e1e3ea40c9a95
Successfully built fire


### Importación de Librerias

In [2]:
#de Videos a Imagenes.
import cv2
from google.colab import drive

#Extracción de emociones
import pandas as pd
from deepface import DeepFace

# Cantidad de imagenes generadas.
import os
import glob

# Monta la unidad de Google Drive
drive.mount('/content/drive')

Directory  /root /.deepface created
Directory  /root /.deepface/weights created
Mounted at /content/drive


### De Video a Imagenes, de Imagenes a Emociones ;)

In [3]:
# Especifíca la ruta de la carpeta que contiene los videos
ruta_carpeta_videos = '/content/drive/MyDrive/0 New/Validacion DeepFace/DFEW Database/part_1/'  # Cambia esta ruta a la ruta de tu carpeta en Google Colab

# Usamos el módulo glob para buscar archivos con una extensión específica (en este caso, videos con extensión .mp4)
ruta_archivos_videos = os.path.join(ruta_carpeta_videos, '*.mp4')
lista_archivos_videos = glob.glob(ruta_archivos_videos)

# Contamos la cantidad de archivos encontrados  --- m es la cantidad de videos analizadas en el estudio.
m = len(lista_archivos_videos)  # Número de videos en la carpeta.

# Imprimimos el resultado
print(f'La cantidad de archivos de videos en la carpeta es: {m}')

# Creamos la base de datos que almacenará los videos y sus resultados.
df_BD = pd.DataFrame(columns=['Nombre de Imagen', 'angry', 'disgust', 'fear', 'sad', 'neutral', 'surprise', 'happy', 'Video', 'Q_images'])


# re-Asignamos Manualmente la cantidad de videos que queremos extraer. 
#m = 1000

# Loop sobre cada video: Extracción de frames del Video y Computación Afectiva de los frames.
for a in range (1, m+1):    # m es la cantidad de videos.-

    # Abrimos cada video en Google Drive, vamos cambiando el final del nombre del archivo que es un correlativo numérico.
    numero_video = a
    ruta_video = "/content/drive/MyDrive/0 New/Validacion DeepFace/DFEW Database/part_1/" + str(numero_video) + ".mp4"

    # Hacemos la función para extraer y almacenar las imagenes en la carpeta temporal del google Colab (sample data).
    def getFrame(sec):
        vidcap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
        hasFrames, image = vidcap.read()
        if hasFrames:
            cv2.imwrite("sample_data/image" + str(count) + ".jpg", image)     # Guardar el frame como archivo JPG
        return hasFrames

    # Abrir el video desde Google Drive
    vidcap = cv2.VideoCapture(ruta_video)

    # Obtener los FPS del video
    fps = vidcap.get(cv2.CAP_PROP_FPS)
    print("FPS del video:", fps)

    # Calculamos de framerate en base a los fps del video.
    frameRate = 1 / fps


    #Hacemos la extraccion y almacenamiento de imagenes.
    sec = 0                     # Segundo desde donde empezamos a extraer
    count = 1
    success = True
    while success:
        sec = sec + frameRate
        sec = round(sec, 2)
        success = getFrame(sec)
        count = count + 1
    # Liberar el objeto VideoCapture y finalizar
    vidcap.release()
    cv2.destroyAllWindows()

    # COMPUTACION AFECTIVA de los frames del Video.
    # Especifica la ruta de la carpeta que contiene las imágenes
    ruta_carpeta = '/content/sample_data'  # Cambia esta ruta a la ruta de tu carpeta en Google Colab

    # Usamos el módulo glob para buscar archivos con una extensión específica (en este caso, imágenes con extensión .jpg)
    ruta_archivos = os.path.join(ruta_carpeta, '*.jpg')
    lista_archivos = glob.glob(ruta_archivos)

    # Contamos la cantidad de archivos encontrados  --- N es la cantidad de imagenes analizadas en el video.
    n = len(lista_archivos) #numero de imagenes en la carpeta.

    # Imprimimos el resultado
    print(f'La cantidad de archivos de imágenes en la carpeta es: {n}')

    # --------------------------------------------------------------------------------------------
    #EXTRACCION DE EMOCIONES A PARTIR DE IMAGENES.
    # Crear una lista para almacenar los datos
    data = []

    # Bucle para analizar cada imagen y guardar los datos en la lista (data)
    for i in range(1, n+1):     # n es la cantidad de archivos encontrados en la carpeta con formato de imagen (son los frames extraídos).
        # Analizamos la imagen con DeepFace
        img_path = f"sample_data/image{i}.jpg"  # Ruta de la imagen
        objs = DeepFace.analyze(img_path=img_path, actions=['emotion'], enforce_detection=False)
        
        # Obtener el nombre de la imagen y las emociones analizadas
        nombre_imagen = f"imagen{i}.jpg"
        emociones = objs[0]['emotion']
        
        # Agregar los datos a la lista
        data.append([nombre_imagen, emociones])
        
    # Crear un DataFrame de pandas a partir de la lista de datos
    df = pd.DataFrame(data, columns=['Nombre de Imagen', 'Emociones'])

    # Aplicar una función para convertir el diccionario de emociones en columnas individuales (una lista de varios elementos con valores a varias columnas).
    df = df.join(df['Emociones'].apply(pd.Series))

    # Eliminar la columna 'Emociones' original
    df = df.drop('Emociones', axis=1)

    # Redondear todos los valores en el DataFrame a un decimal.
    df = df.round(1)

    # Lista con el nuevo orden de las columnas.
    nuevo_orden = ['Nombre de Imagen', 'happy', 'sad', 'neutral', 'angry',  'surprise', 'disgust', 'fear' ] #'angry', 'disgust', 'fear', 'sad', 'neutral', 'surprise', 'happy']

    # Reordenamos las columnas en el DataFrame.
    df = df[nuevo_orden]

    # Agregamos el identificador del video.
    df=df.assign(Video=numero_video)

    # Agregamos columna que indica cantidad de imagenes extraidas del Video.
    df=df.assign(Q_images=n)

    # Agregamos dataframe de emociones del video (df) al dataframe que almacena los resultados de todos los videos (df2).
    df_BD = df_BD.append(df)

    # Eliminar archivos jpg de la carpeta. nombre carpeta. Con esto terminamos el loop.
    # Ruta a la carpeta que contiene las imágenes JPG
    ruta_carpeta = '/content/sample_data/'

    # Obtener la lista de archivos en la carpeta.
    archivos = os.listdir(ruta_carpeta)

    # Iterar sobre los archivos y eliminar las imágenes en formato JPG.
    for archivo in archivos:
        if archivo.endswith('.jpg'):
            ruta_archivo = os.path.join(ruta_carpeta, archivo)
            os.remove(ruta_archivo)

#Imprimimos el resultado
df_BD

La cantidad de archivos de videos en la carpeta es: 1603
FPS del video: 23.976
La cantidad de archivos de imágenes en la carpeta es: 23
facial_expression_model_weights.h5 will be downloaded...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: /root/.deepface/weights/facial_expression_model_weights.h5
100%|██████████| 5.98M/5.98M [00:00<00:00, 64.8MB/s]
Action: emotion: 100%|██████████| 1/1 [00:00<00:00,  6.39it/s]


FPS del video: 23.976
La cantidad de archivos de imágenes en la carpeta es: 11


Action: emotion: 100%|██████████| 1/1 [00:00<00:00, 11.55it/s]


FPS del video: 23.976
La cantidad de archivos de imágenes en la carpeta es: 19


...

Action: emotion: 100%|██████████| 1/1 [00:00<00:00,  8.91it/s]


FPS del video: 23.976
La cantidad de archivos de imágenes en la carpeta es: 7


Action: emotion: 100%|██████████| 1/1 [00:00<00:00, 10.09it/s]


FPS del video: 0.0
La cantidad de archivos de imágenes en la carpeta es: 0


ValueError: ignored

### Guardamos la BD

In [4]:
#Guardamos la Base de datos.
ruta_video

'/content/drive/MyDrive/0 New/Validacion DeepFace/DFEW Database/part_1/1601.mp4'

In [5]:
import pandas as pd

# Guardamos el DataFrame df2.
# Ruta de destino en Google Colab
ruta_destino = '/content/sample_data/df_BD.csv'
# Guardar el DataFrame como archivo CSV
df_BD.to_csv(ruta_destino, index=False)
# Ruta de destino en Google Colab 
ruta_destino = '/content/sample_data/df_BD.xlsx'
# Guardar el DataFrame como archivo Excel
df_BD.to_excel(ruta_destino, index=False)

In [7]:
#copiamos la dataframe en df2.
df2 = df_BD.copy()

### Promedios de emocion por Video.

In [9]:
### Promedios de emocion por Video.
# Columnas de emociones
columnas_emociones = ['happy', 'sad', 'neutral', 'angry',  'surprise', 'disgust', 'fear' ]
# Sumar los valores de las columnas de emociones para cada video y la divide por la cantidad de imagenes del video (valor de cualquier casilla de Q_imagenes del video)
df_mean = df2.groupby('Video')[columnas_emociones].mean().reset_index()
print(df_mean)

      Video      happy        sad    neutral      angry   surprise   disgust  \
0         1   0.000000  43.452174   2.413043   0.026087  14.447826  0.000000   
1         2   0.000000  52.636364  39.290909   0.418182   0.309091  0.000000   
2         3   0.000000  74.931579  13.136842   0.000000   0.000000  0.000000   
3         4   0.000000  91.228571   0.814286   0.842857   0.014286  0.000000   
4         5   0.000000  50.630000  11.370000  28.760000   1.110000  0.000000   
...     ...        ...        ...        ...        ...        ...       ...   
1595   1596  39.245455   0.000000   0.000000  45.663636   0.000000  0.000000   
1596   1597   5.687500  23.375000  29.437500  18.212500   2.537500  0.000000   
1597   1598  59.480000   2.000000   0.000000  38.220000   0.000000  0.000000   
1598   1599  31.257143  11.700000  12.771429   3.014286   0.385714  0.271429   
1599   1600   0.257143  79.014286   0.657143   6.800000   1.614286  0.000000   

           fear  
0     39.656522  
1  

### Conteo  //  Contar Casos con prob mayor a un umbral. // 

In [13]:
# Columnas de emociones
columnas_emociones = ['happy', 'sad', 'neutral', 'angry',  'surprise', 'disgust', 'fear' ]
# Umbral para filtrar los valores de las emociones
umbral = 60
# Aplicar la máscara booleana y contar las veces que se supera el umbral para cada video
df_cont_u60 = df2[columnas_emociones].applymap(lambda x: 1 if x > umbral else 0).groupby(df2['Video']).sum().reset_index()
# Imprimir el DataFrame de resultados
df_cont_u60

,Video,happy,sad,neutral,angry,surprise,disgust,fear
0,1,0,11,0,0,3,0,5
1,2,0,3,3,0,0,0,1
2,3,0,13,2,0,0,0,1
3,4,0,7,0,0,0,0,0
4,5,0,5,1,2,0,0,0
...,...,...,...,...,...,...,...,...
1595,1596,3,0,0,4,0,0,0
1596,1597,0,2,2,2,0,0,2
1597,1598,3,0,0,2,0,0,0
1598,1599,2,0,0,0,0,0,1


### Respaldamos la BD del Resumen de las observaciones de los videos en formato csv y xlsx.

In [15]:
# Guardamos la BD en .csv y .xlsx .-
import pandas as pd

# Ruta de destino en carpeta temporal de Google Colab sample_data
ruta_destino = '/content/sample_data/nuevo_dataframedf_BD.csv'

# Guardar el DataFrame como archivo CSV
nuevo_dataframe.to_csv(ruta_destino, index=False)

# Ruta de destino en Google Colab
ruta_destino = '/content/sample_data/nuevo_dataframedf_BD.xlsx'

# Guardar el DataFrame como archivo Excel
nuevo_dataframe.to_excel(ruta_destino, index=False)

### Eliminamos valores perdidos o NaNs

In [25]:
# Eliminar los valores NaN en la columna 'Columna1'.
nuevo_dataframe_sin_nans = nuevo_dataframe2.dropna(subset=['happy_mean', 'sad_mean','happy_sum', 'neutral_mean', 'angry_mean', 'surprise_mean', 'disgust_mean', 'fear_mean', ])
# Mostrar el DataFrame sin los valores NaN.
nuevo_dataframe_sin_nans.head(5)

,Video,happy_sum,sad_sum,neutral_sum,angry_sum,surprise_sum,disgust_sum,fear_sum,happy_mean,sad_mean,...,disgust_cont_u60,fear_cont_u60,1happy,2sad,3neutral,4angry,5surprise,6disgust,7fear,label
0,1,0.0,999.4,55.5,0.6,332.3,0.0,912.1,0.0,43.452174,...,0.0,5.0,0,6,4,0,0,0,0,2
1,2,0.0,579.0,432.2,4.6,3.4,0.0,80.7,0.0,52.636364,...,0.0,1.0,0,2,8,0,0,0,0,3
2,3,0.0,1423.7,249.6,0.0,0.0,0.0,226.5,0.0,74.931579,...,0.0,1.0,3,1,1,0,5,0,0,0
3,4,0.0,638.6,5.7,5.9,0.1,0.0,49.6,0.0,91.228571,...,0.0,0.0,0,6,3,1,0,0,0,2
4,5,0.0,506.3,113.7,287.6,11.1,0.0,81.2,0.0,50.630000,...,0.0,0.0,0,2,2,1,4,0,1,0
